# Init Lakehouse

In [20]:
%%configure -f
{
    "defaultLakehouse": {"name": "DE_LH_100_BondedWarehouse"}
}

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, -1, Finished, Available, Finished)

# Init Imports (these need cutting-down post creation)

In [21]:
import os
import csv
import re
import shutil
import unicodedata
import pandas as pd

import notebookutils

#from decimal import Decimal
from datetime import datetime
from datetime import timedelta
#from collections import Counter
#from functools import reduce
import time

#from pyspark import StorageLevel
from pyspark.sql import DataFrame, Row
from pyspark.sql.functions import col, lit, when, concat, concat_ws, coalesce, count, monotonically_increasing_id, sum, to_date, udf, current_timestamp, length, substring, split, size, asc, row_number
from pyspark.sql.functions import broadcast, hash, array, expr, array_distinct, date_format
from pyspark.sql.types import *
#from pyspark.sql import Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 3, Finished, Available, Finished)

# Init Export Process

In [22]:
def save_dataframe_to_csv(df, file_path, show_header=False, mode='overwrite'):
    """
    Save a DataFrame as a single CSV file in a PySpark application.

    Parameters:
    df (pyspark.sql.DataFrame): The DataFrame to save.
    file_path (str): The path to save the CSV file.
    header (bool): Whether to include the header in the CSV file. Default is True.
    mode (str): The write mode. Options are 'overwrite', 'append', 'ignore', 'error' or 'errorifexists'. Default is 'overwrite'.

    Returns:
    None
    """

    use_pipes = len(df.columns) != 1
    print(f'Add pipes: {use_pipes}')
    print(f'Show headers: {show_header}')

    pandas_df = df.toPandas()
    
    # Replace newlines and carriage returns
    pandas_df = pandas_df.replace({r'\r\n': ' ', r'\n': ' ', r'\r': ' '}, regex=True)

    # Create a string representation of the DataFrame with '|' as separator
    # Escape special characters such as commas and pipes
    if use_pipes:
        csv_data = pandas_df.to_csv(sep="|", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")
    else:
        csv_data = pandas_df.to_csv(sep="~", index=False, header=show_header, quoting=csv.QUOTE_NONE, escapechar="\\")

    # Add trailing pipe '|' at the end of each line
    csv_data_with_pipe = '\n'.join([line + '|' for line in csv_data.split('\n') if line])

    # Write to the file
    with open(file_path, 'w') as f:
        f.write(csv_data_with_pipe)


StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 4, Finished, Available, Finished)

# Init Debug & Incremental Vars

In [23]:
workspace_name = notebookutils.mssparkutils.env.getWorkspaceName()

if "DEV" in workspace_name.upper():
    debug = True
    incremental_run = False
    default_days_lag: int = 7

elif "UAT" in workspace_name.upper():
    debug = True
    incremental_run = True
    default_days_lag: int = 7

else:
    debug = False
    incremental_run = True
    default_days_lag: int = 1

if debug:
    print(debug , incremental_run)

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 5, Finished, Available, Finished)

True False


# Init Days Lag Var

In [24]:
filterdate_pipe = ''

#default_days_lag: int = 1

enable_string_truncation = True
create_hash_cols: bool = False
transfer_file: bool = False
retain_error_records_in_ouput_file: bool = False

# Override Debug

#debug = False   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#debug = True   #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

# Override Full Run 

#incremental_run = False    #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<
#incremental_run = True     #<<<<<<<<<<<<<<<<<<<<<<<<<<<  <<<<<<<<<<<<<<<<<<<<

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 6, Finished, Available, Finished)

In [25]:
filterdate = datetime.now() - timedelta(days=default_days_lag)
filterdate = filterdate.date()

if debug:
    print(f'Get Registrations from: {filterdate}')

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 7, Finished, Available, Finished)

Get Registrations from: 2025-05-05


# Init Query(s)

In [26]:
registrations_df = spark.sql("""
SELECT
vendpackingslipjour.Id -- THIS IS FOR RECORDING TRACKING ONLY - REMOVE LATER IN NOTEBOOK
	,'ENDCTG' AS Company_Code
	,'' AS Site_Code
	,vendpackingslipjour.deliverydate AS Registration_Date
	,'U' AS Stock_Detail_Key_Type

	,CASE
		WHEN whsloadline.loadid IS NOT NULL AND whsloadline.itemid IS NOT NULL THEN CONCAT(whsloadline.loadid, '/', whsloadline.itemid)
		WHEN whsloadline.loadid IS NOT NULL THEN whsloadline.loadid
		WHEN whsloadline.itemid IS NOT NULL THEN whsloadline.itemid
		ELSE NULL
	 END AS Stock_Detail_Key

	,vendpackingsliptrans.itemid AS Product_Code
	,CAST(vendpackingsliptrans.qty AS DECIMAL(10,2)) AS Quantity

FROM vendpackingslipjour

INNER JOIN vendpackingsliptrans
    ON vendpackingslipjour.recid = vendpackingsliptrans.vendpackingslipjour
    AND vendpackingslipjour.dataareaid = vendpackingsliptrans.dataareaid
	
-- INNER JOIN inventdim
--     ON vendpackingsliptrans.InventDimId = inventdim.InventDimId

INNER JOIN purchtable
    ON vendpackingslipjour.purchid = purchtable.purchid
    AND vendpackingslipjour.dataareaid = purchtable.dataareaid

INNER JOIN whsloadline
    ON vendpackingsliptrans.inventdimid = whsloadline.inventdimid
    AND vendpackingsliptrans.inventtransid = whsloadline.inventtransid
    AND vendpackingsliptrans.itemid = whsloadline.itemid
    AND vendpackingsliptrans.dataareaid = whsloadline.dataareaid

INNER JOIN whsloadtable
    ON whsloadtable.loadid = whsloadline.loadid
    AND whsloadtable.dataareaid = whsloadline.dataareaid

WHERE whsloadtable.hslcomminvbookingreference != ''
AND purchtable.inventsiteid = 'PAR' -- Should be inventdim.InventLocationId = 'PAR' waiting for table sync to Dataverse
AND vendpackingslipjour.dataareaid IN ('end.', 'END.')

"""
)
if debug:
      display(registrations_df)

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a869e216-9d16-4572-a3a7-3cef65e5df92)

In [27]:
registrations_df = registrations_df.select(
    substring(col("Id").cast("string"),1, 99).alias("Id"), # REMOVE THIS LATER - ONLY FOR TRACKING
    substring(col("Company_Code").cast("string"),1, 10).alias("Company_Code"),
    substring(col("Site_Code").cast("string"),1, 4).alias("Site_Code"),
    col("Registration_Date").cast("date").alias("Registration_Date"),
    substring(col("Stock_Detail_Key_Type").cast("string"),1, 1).alias("Stock_Detail_Key_Type"),
    substring(col("Stock_Detail_Key").cast("string"),1, 50).alias("Stock_Detail_Key"),
    substring(col("Product_Code").cast("string"),1, 25).alias("Product_Code"),
    substring(col("Quantity").cast("string"),1, 11).alias("Quantity")
)
if debug:
    display(registrations_df)

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, acbb3130-2725-4712-bae1-2f242c2e3949)

## Date Field Changing Post Query(s)

In [28]:
list_date_columns_1 = [name for name, dtype in registrations_df.dtypes if dtype in ('date','timestamp')]

if debug:
    print("Date Columns to change: " , list_date_columns_1)

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 10, Finished, Available, Finished)

Date Columns to change:  ['Registration_Date']


In [29]:
for column in list_date_columns_1:
    registrations_df = registrations_df.withColumn(column, date_format(column, "dd-MM-yyyy"))

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 11, Finished, Available, Finished)

In [30]:
if debug:
    display(registrations_df)

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 12, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 18d81eb3-905e-4e39-b117-b99463a362e8)

# Init Good File Name & Date Logic

In [31]:
file_path_folder = "/lakehouse/default/Files/Output/"
file_extention = '.dat'

# file date time - stamp tomorrow's date if after 6.15pm --Nick: had to knock it back 1 hour to account for timezone difference; working
now = datetime.now()
cutoff_time = now.replace(hour=17, minute=15, second=0, microsecond=0)

if now > cutoff_time:
    tomorrow = now + timedelta(days=1)
    #file_datetime = tomorrow.strftime('%Y-%m-%d')
    file_datetime = tomorrow.strftime('%Y-%m-%d-%H')
else:
    file_datetime = now.strftime('%Y-%m-%d-%H')


file_name = "registrations" + "_" + file_datetime + file_extention
file_path = file_path_folder + file_name

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 13, Finished, Available, Finished)

In [32]:
if debug:
    print("Now: " , now)
    print("Cutoff: " , cutoff_time)
    print("File Date: " , file_datetime)
    print("File Folder Path: " , file_path_folder)
    print("Good File Name: " , file_name)
    print("Good File Name: " , file_path)

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 14, Finished, Available, Finished)

Now:  2025-05-12 08:54:10.485349
Cutoff:  2025-05-12 17:15:00
File Date:  2025-05-12-08
File Folder Path:  /lakehouse/default/Files/Output/
Good File Name:  registrations_2025-05-12-08.dat
Good File Name:  /lakehouse/default/Files/Output/registrations_2025-05-12-08.dat


# Remove Rows Already Sent

In [33]:
if incremental_run:
    
    # REMOVE ROWS FROM CURRENT RUN THAT HAVE ALREADY BEEN SENT (EXIST IN RECORD TRACKING)

    lakehouse_table_name = "bondedwarehouserecordtracking_registrations"
    container_column = "Id"

    try:
        # Loads already sent Containers from record tracking
        sentrecords_df = spark.read.table(lakehouse_table_name).select(container_column).distinct()

        # Filter each input dataframe to EXCLUDE already sent
        registrations_df = registrations_df.join(sentrecords_df, registrations_df["Id"] == sentrecords_df["Id"], "left_anti")

    except Exception as e:
        print(f"An error occurred: {e}")

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 15, Finished, Available, Finished)

# Export Good

In [34]:
final_good_df = registrations_df.drop("Id") # use this for output file and use registrations_df for record tracking

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 16, Finished, Available, Finished)

In [35]:
save_dataframe_to_csv(final_good_df, file_path, show_header=False)

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 17, Finished, Available, Finished)

Add pipes: True
Show headers: False


In [36]:
if final_good_df.take(1):

    ready_to_copy = True

else:

    ready_to_copy = False

if debug:
    print(ready_to_copy)

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 18, Finished, Available, Finished)

False


# Init Record Tracking

In [37]:
if incremental_run:

    # Save the final_df to different tables based on the exportfile variable value

    def record_tracking_df_to_table(dataframe, table_name, file_name):
        """Saves distinct records to a table, checking for duplicates and enabling column mapping."""
        table_name_lower = table_name.lower()

        # Check if table exists
        table_exists = True
        try:
            spark.read.table(table_name_lower)
            print(f"Table {table_name_lower} exists.")
        except Exception as e:
            print(f"Table {table_name_lower} does not exist.")
            table_exists = False

        # Columns to deduplicate on (excluding metadata)
        dedup_cols = [col for col in dataframe.columns if col not in ["Timestamp", "ExportName", "ExportDate"]]

        if table_exists:
            try:
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))

                existing_df = spark.read.table(table_name_lower)

                distinct_existing_df = existing_df.dropDuplicates(subset=dedup_cols)
                initial_existing_count = distinct_existing_df.count()
                print(f"Existing distinct count: {initial_existing_count}")

                distinct_new_df = dataframe.dropDuplicates(subset=dedup_cols)

                combined_distinct_df = distinct_new_df.unionByName(distinct_existing_df) \
                                                    .dropDuplicates(subset=dedup_cols)
                final_distinct_count = combined_distinct_df.count()
                print(f"Final distinct count: {final_distinct_count}")

                rows_added = final_distinct_count - initial_existing_count
                print(f"Added {rows_added} new distinct records to {table_name_lower}.")

                combined_distinct_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

            except Exception as e:
                print(f"Exception: Saving all records in new table. Exception: {e}")
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")

        else:
            try:
                print(f"Table doesn't exist. Saving all records in new table.")
                dataframe = dataframe.withColumn("ExportName", lit(file_name).cast(StringType())) \
                                    .withColumn("ExportDate", current_timestamp().cast(TimestampType()))
                dataframe = dataframe.dropDuplicates(subset=dedup_cols)
                dataframe.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name_lower)
                print(f"Distinct records saved to {table_name_lower}.")
            except Exception as e:
                print(f"Error saving data: {e}")

    table_prefix = 'BondedWarehouseRecordTracking_'

    record_tracking_df_to_table(registrations_df, f"{table_prefix}registrations", file_name)

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 19, Finished, Available, Finished)

Table bondedwarehouserecordtracking_registrations exists.
Existing distinct count: 299
Final distinct count: 299
Added 0 new distinct records to bondedwarehouserecordtracking_registrations.
Distinct records saved to bondedwarehouserecordtracking_registrations.


# Init Send To Azure Blob Storage

In [38]:
if ready_to_copy == False:
    output_msg = f'Process Complete'

    notebookutils.notebook.exit(output_msg)

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, 20, Finished, Available, Finished)

ExitValue: Process Complete

## Copy the file to an ADLS account for loading to the SFTP
Set the source and destination paths

In [ ]:
if "DEV" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_dev/ToBeSent/" + file_name
    
elif "UAT" in workspace_name.upper():
    dest_abfss_file_path = "Files/bonded_warehouse_uat/ToBeSent/" + file_name

else:
    dest_abfss_file_path = "Files/bonded_warehouse/ToBeSent/" + file_name

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, -1, Cancelled, , Cancelled)

In [ ]:
source_abfss_file_path = 'Files/Output/' + file_name

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, -1, Cancelled, , Cancelled)

In [ ]:
if transfer_file:
    notebookutils.fs.fastcp(source_abfss_file_path, dest_abfss_file_path)

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, -1, Cancelled, , Cancelled)

In [ ]:
if ready_to_copy == True:
    output_msg = f'Process Complete'

notebookutils.notebook.exit(output_msg)

StatementMeta(, 5d40d718-eeae-4c98-8d0f-b039cc468475, -1, Cancelled, , Cancelled)